# Guided Exercise: Support Router

This notebook combines the repository concepts into one local support-routing workflow. You will estimate token pressure, retrieve lesson candidates with the local sparse index, apply a confidence threshold, and explain the decision.

The workflow is intentionally local and auditable. Every intermediate value stays visible: token count, estimated cost, top-k candidates, similarity scores, threshold decision, and fallback path.

## Learning Objectives

By the end of this guided exercise, you should be able to:

- Inspect the data needed for a retrieval-based support router.
- Configure token, cost, and confidence assumptions.
- Implement a routing helper that returns structured evidence.
- Tune a threshold using in-scope and out-of-scope questions.
- Explain the router's value and limitations to a stakeholder.

## Router Decision Flow

```mermaid
graph LR
    A["Support<br>question"] --> B["Estimate tokens<br>and cost"]
    B --> C["Retrieve top-k<br>lesson candidates"]
    C --> D["Inspect<br>top score"]
    D --> E{"Top score above<br>threshold?"}
    E -- Yes --> F["Use retrieved<br>candidates"]
    E -- No --> G["Fallback or<br>ask for clarification"]
    F --> H["Return evidence"]
    G --> H
```

The key habit is evidence-based routing. A decision should include the input, token estimate, cost estimate, top candidates with scores, threshold decision, and short explanation.

## Implementation Helpers

The previous notebooks showed the tokenization, sparse-vector, and retrieval mechanics. This exercise imports the shared helpers so the notebook can focus on the routing tasks rather than repeating the same implementation stack.

In [ ]:
import csv
from pathlib import Path

from retrieval_helpers import (
    LocalRetrievalIndex,
    count_tokens,  # noqa: F401 - used when completing the exercise.
    estimate_request_cost,  # noqa: F401 - used when completing the exercise.
    format_usd,
)

## Task 1: Inspect The Data

Load the course catalog and support questions. Check the fields before building the router, because missing metadata makes retrieval and evaluation hard to debug.

In [ ]:
catalog_path = Path("data") / "course_catalog.csv"
questions_path = Path("data") / "support_questions.csv"

with catalog_path.open(newline="", encoding="utf-8") as file:
    catalog = list(csv.DictReader(file))
with questions_path.open(newline="", encoding="utf-8") as file:
    questions = list(csv.DictReader(file))

print(f"Catalog lessons: {len(catalog)}")
print(f"Support questions: {len(questions)}")
print("Catalog columns:", list(catalog[0]))
print("Question columns:", list(questions[0]))

### Reference Answer

The catalog should expose `lesson_id`, `title`, `module`, and `description`. The support questions should expose `question_id`, `question`, `expected_topic`, and `difficulty`. If those fields are missing, the router cannot explain or evaluate retrieval quality reliably.

## Task 2: Configure The Router

These constants are product assumptions, not universal truths. Retrieved context increases input tokens, expected output tokens reserve answer space, token prices make cost visible, and the threshold controls fallback behavior.

In [ ]:
index = LocalRetrievalIndex.from_csv(catalog_path)
TOP_K = 3
TOKENS_PER_RETRIEVED_LESSON = 120
EXPECTED_OUTPUT_TOKENS = 180
INPUT_PRICE_PER_MILLION = 0.50
OUTPUT_PRICE_PER_MILLION = 1.50
CONFIDENCE_THRESHOLD = 0.15

retrieved_context_tokens = TOP_K * TOKENS_PER_RETRIEVED_LESSON
print("Retrieved context tokens:", retrieved_context_tokens)
print("Confidence threshold:", CONFIDENCE_THRESHOLD)

## Task 3: Implement The Routing Helper

Complete the helper so it returns one structured decision per question. The output should include:

- the original question,
- approximate question tokens,
- estimated input tokens,
- estimated request cost,
- top retrieval candidates,
- final decision: `use retrieved candidates` or `fallback`.

In [ ]:
# @TODO: Implement a `route_question(question)` helper below.
# Hints:
# - count question tokens with count_tokens(question)
# - add retrieved_context_tokens to estimate input size
# - call estimate_request_cost(...)
# - retrieve candidates with index.search(question, k=TOP_K)
# - compare the top score with CONFIDENCE_THRESHOLD


def route_question(question):
    pass

<details>
  <summary>Show Reference Solution</summary>

```python
def route_question(question: str) -> dict:
    question_tokens = count_tokens(question)
    input_tokens = question_tokens + retrieved_context_tokens
    cost = estimate_request_cost(
        input_tokens=input_tokens,
        output_tokens=EXPECTED_OUTPUT_TOKENS,
        input_price_per_million=INPUT_PRICE_PER_MILLION,
        output_price_per_million=OUTPUT_PRICE_PER_MILLION,
    )
    results = index.search(question, k=TOP_K)
    top_score = results[0].score
    decision = (
        "use retrieved candidates" if top_score >= CONFIDENCE_THRESHOLD else "fallback"
    )
    return {
        "question": question,
        "question_tokens": question_tokens,
        "input_tokens": input_tokens,
        "estimated_cost": cost,
        "results": results,
        "top_score": top_score,
        "decision": decision,
    }
```

</details>


## Task 4: Route Representative Questions

Use one tokenization question, one retrieval question, and one unrelated question that should trigger fallback behavior.

In [ ]:
sample_questions = [
    "How do I know if my prompt will fit into the context window?",
    "Can I search course notes by meaning instead of exact keywords?",
    "How do I make vegan bolognese?",
]

if route_question(sample_questions[0]) is None:
    print(
        "Complete `route_question` in the practice cell above, then re-run this cell."
    )
else:
    for question in sample_questions:
        routed = route_question(question)
        print(f"\nQuestion: {routed['question']}")
        print(f"Question tokens: {routed['question_tokens']}")
        print(
            f"Estimated input tokens with retrieved context: {routed['input_tokens']}"
        )
        print(
            f"Estimated request cost: {format_usd(routed['estimated_cost'].total_cost)}"
        )
        print(f"Decision: {routed['decision']}")
        for result in routed["results"]:
            print(f"  {result.rank}. {result.title} | score={result.score:.3f}")

### Reference Answer

The two in-scope questions should retrieve course modules related to context windows or retrieval. The unrelated question should fall back when the top score is below the chosen threshold. If it does not, the threshold is too permissive for this support workflow.

## Task 5: Tune The Threshold

Try multiple thresholds and inspect what changes. Do not tune only for the unrelated question; also check whether valid course questions are accidentally rejected.

This comparison does not update the router. Set `CONFIDENCE_THRESHOLD` in Task 2 to your chosen value, re-run that cell, then re-run Task 4 before testing your own questions in Task 6.

In [ ]:
for threshold in [0.10, 0.25, 0.40, 0.50]:
    print(f"\nThreshold: {threshold}")
    for question in sample_questions:
        top = index.search(question, k=TOP_K)[0]
        decision = "use" if top.score >= threshold else "fallback"
        print(f"  {decision:8} | {top.score:.3f} | {question}")

### Reference Answer

A useful threshold separates weak unrelated matches from normal in-scope questions. If increasing the threshold rejects valid questions, the setting is too strict. If decreasing it accepts unrelated questions, the setting is too permissive.

In a real project, tune thresholds with labeled examples, review false accepts and false fallbacks, and monitor the fallback rate after release.

## Task 6: Add Your Own Test Questions

Create 3 new support questions:

1. one clearly in scope,
2. one ambiguous but course-related,
3. one unrelated.

Run them through the router and record whether the decision makes sense.

In [ ]:
new_questions = [
    # @TODO: Create 3 new questions to test the router.
]

if route_question(sample_questions[0]) is None:
    print(
        "Complete `route_question` in the practice cell above, then re-run this cell."
    )
elif not new_questions:
    print("Add your 3 test questions to `new_questions` above, then re-run this cell.")
else:
    for question in new_questions:
        routed = route_question(question)
        print(f"\n{question}")
        print(f"Decision: {routed['decision']} | top score: {routed['top_score']:.3f}")
        for result in routed["results"]:
            print(f"  {result.rank}. {result.title} | score={result.score:.3f}")

<details>
  <summary>Show Examples</summary>

```python
new_questions = [
    "How can I estimate the cost of a long RAG answer?",
    "Why did my retrieval result look strange?",
    "Why is my dog eating grass?",
]
```

</details>


## Task 7: Explain The System

Write a stakeholder explanation that mentions what the router does, the role of token count, how retrieved lessons are used, the reason fallback exists, and what the router cannot guarantee.

In [ ]:
stakeholder_summary = """
# @TODO: Fill in the stakeholder summary.
"""
print(stakeholder_summary.strip())

### Reference Answer

A clear stakeholder explanation should mention the system input, the token and cost estimate, the retrieval step, the confidence threshold, and the reason fallback exists. It should avoid claiming that sparse retrieval proves correctness or fully understands meaning.

<details>
  <summary>Show Reference Answer</summary>

```python
stakeholder_summary = """
This helper estimates the size and approximate cost of a support question, retrieves the most similar lesson descriptions, and flags low-confidence matches for fallback. It can help route questions to useful course material, but it can still be wrong because similarity scores are relative to the indexed documents and the sparse vectorizer may miss meaning or over-weight shared words. In a real application, I would monitor top-k relevance, low-confidence fallback rate, and user feedback on bad matches.
"""
```

</details>


## What To Inspect

Focus on the decision, not only the top title. A retrieved lesson can look plausible while the score is weak or the query is unrelated. For each routed question, inspect:

- the top score,
- whether the top-3 candidates are useful,
- whether fallback would be safer,
- whether a stakeholder could understand the decision from the evidence.

## Extension Ideas

- Add a `source_url` column to the catalog and print citations with each result.
- Add five new support questions and compute HitRate@3 again.
- Add a separate fallback message for unrelated questions versus ambiguous course questions.
- Log each routed question to a small CSV file, then summarize fallback rate and average estimated cost.
- Replace the local sparse vectorizer with a real embedding model only after the local evaluation flow is clear.

These extensions should preserve the same debugging habit: every answer should be traceable back to input, retrieved context, score, threshold, and decision.

## Checklist

Before finishing, make sure you can explain:

- the difference between a token and a word,
- why retrieved context changes cost,
- why local sparse retrieval can return weak matches,
- how dense embeddings differ from this local sparse index,
- which threshold you would choose and why,
- one risk to monitor before using this inside an LLM application.